# 09 — Quantized Multi-Hop Reasoning on Tiny Aya (3.35B)
## قياس أثر التكميم والاستدلال متعدد الخطوات في النماذج متعددة اللغات أصلاً (Tiny Aya Earth)

> **The Empirical Puzzle & Research Hypothesis:**
> In notebooks `07` and `08`, we evaluated Google's `gemma-2-2b` on 2-hop factual reasoning across English, Modern Standard Arabic (MSA), and Egyptian Arabic (Masri). We uncovered two critical vulnerabilities:
> 1. **The L22 English Pivot:** Despite receiving Arabic prompts, Gemma-2 resolves multi-hop factual relations ($[\text{Suez Canal}] \to [\text{Egypt}] \to [\text{Cairo}]$) by routing through an internal English-dominated subspace at Layer 22 ($P(\text{' Cairo'}) = 0.975$) before attempting a late cross-lingual un-embedding projection.
> 2. **Asymmetric Quantization Penalty:** Low-bit weight quantization (Int4 RTN) shattered this late cross-lingual projection, causing a $-65.8\%$ factual drop in MSA and a $-72.9\%$ drop in Masri, with Masri collapsing into a conversational question-mark attractor ($P('?') = 0.994$).
>
> **The Tokenization Nuance & The Hybrid Evaluation Methodology:**
> In Arabic language modeling, the definite article `'ال'` is a high-frequency unigram prefix. Without contextual whitespace, Arabic entity names decompose into subwords (e.g. `['ال', 'قاهرة']`, token IDs `[379, 246588]`). Tracking `'ال'` alone risks measuring generic grammatical prefixing rather than true factual retrieval.
>
> To resolve this with empirical rigor, we implement a **Hybrid Evaluation Framework**:
> 1. **Unified Single-Token Logit Lens:** By structuring the assistant turn to terminate in a natural whitespace-delimited connector (`"The capital is "` / `"العاصمة هي "` / `"عاصمة البلد هي "`), the model naturally predicts the **unified single-token** entity name: `' Cairo'` (ID `147633`) in English, and `' القاهرة'` (ID `97936`) in Arabic. This preserves clean 1-token logit lens tracking across all 36 transformer layers.
> 2. **Candidate City Contrastive Discrimination:** At the final un-embedding layer, we contrast $P(\text{Cairo})$ against a battery of competing Egyptian cities (`Alexandria`, `Giza`, `Suez`, `Aswan`, `Mansoura`, `Tanta`). Demonstrating a high contrast ratio ($P(\text{Cairo}) / P(\text{Alexandria}) \gg 1$) proves genuine factual knowledge retrieval over generic linguistic probability.
>
> **Core Questions for Tiny Aya Earth (3.35B):**
> - **H1 (Direct Arabic Circuit):** Does native multilingual pre-training bypass the English lingua franca pivot, resolving Arabic facts directly within an Arabic semantic subspace?
> - **H2 (Dialectal Quantization Resilience):** Does native multilingual representation confer structural resilience against low-bit quantization (Int8 and Int4), preventing the dialectal collapse seen in Gemma-2-2B?
> - **H3 (RTN vs Distribution-Aware NF4):** Does information-theoretically optimal 4-bit quantization (NormalFloat4) protect fragile dialectal representations where naive Round-To-Nearest (RTN) fails?

### Step 1: Environment Configuration & ROCm Execution Sanity Check

We initialize the execution environment on AMD Strix Halo using ROCm. We configure PyTorch to use `HSA_OVERRIDE_GFX_VERSION=11.0.0` to target the RDNA 3.5 iGPU architecture cleanly, disable tokenizers parallelism to eliminate fork warnings, and assert CUDA/ROCm device availability.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")


PyTorch Version: 2.5.1+rocm6.2
Device: cuda
Device Name: AMD Radeon Graphics
Allocated Memory: 0.00 MB


amdgpu.ids: No such file or directory


### Step 2: Parallel Multi-Hop Queries, Prefill Framing & Tokenizer Diagnostics

To enable rigorous logit lens evaluation on Tiny Aya without definite-article ambiguity, we format our 2-hop reasoning queries using Aya's native chat template with an assistant response prefill ending in a trailing space:
- **English:** `... <|START_RESPONSE|>The capital is `
- **MSA:** `... <|START_RESPONSE|>العاصمة هي `
- **Masri:** `... <|START_RESPONSE|>عاصمة البلد هي `

**Tokenizer Diagnostic Insight:**
Tiny Aya uses a 262,144-vocabulary `Cohere2` BPE tokenizer. Because BPE treats leading whitespace as part of the token:
- Leading space + Cairo (`' القاهرة'`) maps to a **single unified token ID `97936`**.
- Leading space + Cairo in English (`' Cairo'`) maps to a **single unified token ID `147633`**.
- Competing cities decompose into distinct initial tokens (`' الإ'` for Alexandria, `' الس'` for Suez, `' الجيزة'` for Giza).

Below, we construct the prompts and inspect the tokenization diagnostics.

In [2]:
TOKENIZER_NAME = "CohereLabs/tiny-aya-earth"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

RAW_QUERIES = {
    "EN": ("What is the capital of the country where the Suez Canal is located?", "The capital is "),
    "MSA": ("ما هي عاصمة الدولة التي تقع فيها قناة السويس؟", "العاصمة هي "),
    "Masri": ("ايه هي عاصمة البلد اللي فيها قناة السويس؟", "عاصمة البلد هي "),
}

PROMPTS = {}
print("=== Formatted Multi-Hop Chat Prompts with Assistant Prefill ===")
for lang, (q, prefill) in RAW_QUERIES.items():
    msgs = [{"role": "user", "content": q}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prefill
    PROMPTS[lang] = formatted
    tok_ids = tokenizer.encode(formatted, add_special_tokens=False)
    print(f"\n[{lang:<5}] Total Prompt Length: {len(tok_ids)} tokens")
    print(f"Tail: ...{formatted[-60:]!r}")

# Diagnostic verification of target tokens and candidate cities
CANDIDATE_CITIES = {
    "Cairo": " القاهرة",
    "Alexandria": " الإسكندرية",
    "Giza": " الجيزة",
    "Suez": " السويس",
    "Aswan": " أسوان",
    "Mansoura": " المنصورة",
    "Tanta": " طنطا",
}

print("\n=== Tokenizer Diagnostics: Candidate Egyptian Cities ===")
for name, word in CANDIDATE_CITIES.items():
    ids = tokenizer.encode(word, add_special_tokens=False)
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"{name:12} | Word: {repr(word):14} | IDs: {str(ids):25} | Decoded: {decoded}")


=== Formatted Multi-Hop Chat Prompts with Assistant Prefill ===

[EN   ] Total Prompt Length: 383 tokens
Tail: ...'RN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>The capital is '

[MSA  ] Total Prompt Length: 381 tokens
Tail: ...'F_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>العاصمة هي '

[Masri] Total Prompt Length: 381 tokens
Tail: ...'RN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>عاصمة البلد هي '

=== Tokenizer Diagnostics: Candidate Egyptian Cities ===
Cairo        | Word: ' القاهرة'     | IDs: [97936]                   | Decoded: [' القاهرة']
Alexandria   | Word: ' الإسكندرية'  | IDs: [4868, 45797, 2960, 14267] | Decoded: [' الإ', 'سك', 'ند', 'رية']
Giza         | Word: ' الجيزة'      | IDs: [153046]                  | Decoded: [' الجيزة']
Suez         | Word: ' السويس'      | IDs: [2658, 78092]             | Decoded: [' الس', 'ويس']
Aswan        | Word: ' أسوان'       | IDs: [11637, 6291]             | Decoded: [' أس', 'وان']
Mansoura     | Word: ' المنصورة'    | IDs: [9525, 7831